In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
NVIDIA GeForce RTX 2050


In [2]:
import os

# Use only 1 GPU if available. If CUDA is unavailable, Moirai2 will run on CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

MOIRAI2_MODEL_ID = "Salesforce/moirai-2.0-R-small"  # moirai2 only small? https://github.com/SalesforceAIResearch/uni2ts
MOIRAI2_CONTEXT_LENGTH = 1680
MOIRAI2_BATCH_SIZE = 32
MOIRAI2_MODEL_NAME = "Moirai2"

moirai2_module = Moirai2Module.from_pretrained(MOIRAI2_MODEL_ID)

In [3]:
from pprint import pprint
from copy import deepcopy
import os

import pandas as pd
import numpy as np

from utilsforecast.losses import mase
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive

from src.meta.arima._data_reader import ModelIO
from src.chronos_data import ChronosDataset


def moirai2_predict_df(
    history_df,
    predictor,
    prediction_length,
    freq,
    model_col=MOIRAI2_MODEL_NAME,
    future_ds=None,
):
    """Forecast a long dataframe with columns unique_id, ds, y using Moirai2."""
    required_cols = ["unique_id", "ds", "y"]
    missing_cols = [col for col in required_cols if col not in history_df.columns]
    if missing_cols:
        raise ValueError(f"history_df is missing columns: {missing_cols}")

    moirai_input = history_df[required_cols].copy()
    moirai_input["ds"] = pd.to_datetime(moirai_input["ds"])
    moirai_input = moirai_input.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    ids = []
    grouped_inputs = []
    past_targets = []
    for uid, uid_history in moirai_input.groupby("unique_id", sort=False):
        ids.append(uid)
        grouped_inputs.append(uid_history)
        past_targets.append(uid_history["y"].to_numpy(dtype=np.float32))

    moirai_model = getattr(predictor, "prediction_net", predictor)
    quantile_forecasts = moirai_model.predict(past_targets)
    quantile_levels = np.asarray([float(str(q).lstrip("p")) / (100 if str(q).startswith("p") else 1) for q in moirai_model.module.quantile_levels])
    median_idx = int(np.argmin(np.abs(quantile_levels - 0.5)))
    point_forecasts = quantile_forecasts[:, median_idx, :prediction_length]

    frames = []
    future_ds = None if future_ds is None else pd.to_datetime(pd.Series(future_ds))

    for i, (uid, uid_history) in enumerate(zip(ids, grouped_inputs)):
        y_pred = np.atleast_1d(np.asarray(point_forecasts[i]).squeeze())
        if y_pred.ndim > 1:
            y_pred = y_pred[:, 0]

        if future_ds is not None and len(ids) == 1:
            ds_values = future_ds.iloc[: len(y_pred)].to_numpy()
        else:
            last_ds = uid_history["ds"].max()
            ds_values = pd.date_range(
                start=last_ds,
                periods=prediction_length + 1,
                freq=freq,
            )[1 : len(y_pred) + 1]

        frames.append(
            pd.DataFrame(
                {
                    "unique_id": uid,
                    "ds": ds_values,
                    model_col: y_pred[: len(ds_values)],
                }
            )
        )

    return pd.concat(frames, ignore_index=True)


OVERRIDE_DS = False
algorithm = "catboost"
source = "m4_monthly"
FILENAME = f"assets/trained_metaarima_{source}_{algorithm}.joblib.gz"
meta_arima = ModelIO.load_model(FILENAME)

target = "monash_tourism_monthly"

df, horizon, _, freq, seas_len = ChronosDataset.load_everything(target)
train, test = ChronosDataset.time_wise_split(df, horizon)

moirai2_model = Moirai2Forecast(
    module=moirai2_module,
    prediction_length=horizon,
    context_length=MOIRAI2_CONTEXT_LENGTH,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)
moirai2_predictor = moirai2_model.create_predictor(batch_size=MOIRAI2_BATCH_SIZE)

sf_models = [AutoARIMA(season_length=seas_len), SeasonalNaive(season_length=seas_len)]
model_names = ["MetaARIMA", "AutoARIMA", "SeasonalNaive", MOIRAI2_MODEL_NAME]

uids = train["unique_id"].unique().tolist()

results, predictions = [], []
for uid in uids:
    print(uid)

    df_uid_tr = train.query(f'unique_id=="{uid}"').reset_index(drop=True)
    df_uid_ts = test.query(f'unique_id=="{uid}"').reset_index(drop=True)
    if df_uid_ts.isna().any()["y"]:
        continue

    meta_arima.fit(df_uid_tr, freq=freq, seas_length=seas_len)

    fcst_ma = meta_arima.predict(h=horizon)

    sf = StatsForecast(models=deepcopy(sf_models), freq=freq)
    sf.fit(df_uid_tr)

    fcst_aa = sf.forecast(h=horizon)

    fcst_tsfm1 = moirai2_predict_df(
        df_uid_tr,
        predictor=moirai2_predictor,
        prediction_length=horizon,
        freq=freq,
        future_ds=df_uid_ts["ds"],
    )
    fcst_tsfm1 = fcst_tsfm1[["unique_id", "ds", MOIRAI2_MODEL_NAME]]


    if OVERRIDE_DS:
        fcst_ma["ds"] = df_uid_ts["ds"].values
        fcst_aa["ds"] = df_uid_ts["ds"].values
        fcst_tsfm1["ds"] = df_uid_ts["ds"].values

    uid_test = df_uid_ts.merge(fcst_ma, on=["unique_id", "ds"])
    uid_test = uid_test.merge(fcst_aa, on=["unique_id", "ds"])
    uid_test = uid_test.merge(fcst_tsfm1, on=["unique_id", "ds"])


    err = mase(
        df=uid_test, models=model_names, seasonality=seas_len, train_df=df_uid_tr
    )

    pprint(err)

    predictions.append(uid_test)
    results.append(err)
    results_df = pd.concat(results)
    print(results_df.mean(numeric_only=True))
    print(results_df.median(numeric_only=True))

results_df = pd.concat(results)
predictions_df = pd.concat(predictions).reset_index(drop=True)
print(results_df.mean(numeric_only=True))
print(results_df.median(numeric_only=True))

output_dir = "assets/results/moirai2"
os.makedirs(output_dir, exist_ok=True)
results_df.to_csv(f"{output_dir}/scores,{target}.csv", index=False)
predictions_df.to_csv(f"{output_dir}/predictions,{target}.csv", index=False)

T000000
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive   Moirai2
0   T000000   1.372374   1.286285       1.649169  1.086822
MetaARIMA        1.372374
AutoARIMA        1.286285
SeasonalNaive    1.649169
Moirai2          1.086822
dtype: float64
MetaARIMA        1.372374
AutoARIMA        1.286285
SeasonalNaive    1.649169
Moirai2          1.086822
dtype: float64
T000001
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive  Moirai2
0   T000001   0.943842   0.901422       1.249447  1.01377
MetaARIMA        1.158108
AutoARIMA        1.093853
SeasonalNaive    1.449308
Moirai2          1.050296
dtype: float64
MetaARIMA        1.158108
AutoARIMA        1.093853
SeasonalNaive    1.449308
Moirai2          1.050296
dtype: float64
T000002
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive   Moirai2
0   T000002   0.734334   1.561805       0.256721  0.666325
MetaARIMA        1.016850
AutoARIMA        1.249837
SeasonalNaive    1.051779
Moirai2          0.922306
dtype: float64
MetaARIMA        0.943842
Aut